# PYTORCH MASTERY — NOTEBOOK 9
## Chapters 46–52: GRU → NLP with PyTorch → Embeddings → Attention → Transformers → Implement a Transformer From Scratch → NLP Transformer Project

Continuing directly from Notebook 8. This notebook builds, piece by piece, the architecture underlying every modern LLM.


### PYTORCH MASTERY PROGRESS
Current Chapter: 52 | Current Level: Transformers

- [x] Chapters 1–45 *(Notebooks 1–8)*
- [x] GRU
- [x] NLP with PyTorch
- [x] Embeddings
- [x] Attention
- [x] Transformers
- [x] Implement a Transformer From Scratch
- [x] NLP Transformer Project


---
# CHAPTER 46 — GRU

## 1. Concept
GRU (Gated Recurrent Unit) is a **simplified alternative to LSTM** — it achieves a similar goal (mitigating vanishing gradients through time via gating) with **fewer gates and no separate cell state**, making it computationally cheaper and faster to train, at the cost of being marginally less expressive in some cases.

## 2. GRU vs. LSTM — the Structural Comparison
| | LSTM | GRU |
|---|---|---|
| Gates | 3 (forget, input, output) | 2 (reset, update) |
| Separate cell state? | Yes (hidden state + cell state) | No — only a single hidden state |
| Parameters | More (roughly 4x a plain RNN's parameter count per layer) | Fewer (roughly 3x a plain RNN's) |
| Training speed | Slower (more computation per step) | Faster |
| Typical performance | Often marginally better on tasks with very long-range dependencies or larger datasets | Often comparable, sometimes better on smaller datasets (fewer parameters to overfit) |

## 3. `nn.GRU` — Shapes


In [ ]:
import torch
import torch.nn as nn

gru = nn.GRU(input_size=10, hidden_size=20, num_layers=1, batch_first=True)

x = torch.randn(4, 15, 10)   # batch=4, seq_len=15, input_size=10

# GRU returns output PLUS a SINGLE hidden state (unlike LSTM's tuple of TWO states)
output, h_n = gru(x)

print("output shape:", output.shape, " (batch, seq_len, hidden_size)")
print("h_n shape:", h_n.shape, " (num_layers, batch, hidden_size) -- just ONE state, like plain nn.RNN")
print("\nGRU's return signature matches nn.RNN's EXACTLY (a single h_n, not a tuple) --")
print("this is precisely BECAUSE GRU has no separate cell state to track.")


output shape: torch.Size([4, 15, 20])  (batch, seq_len, hidden_size)
h_n shape: torch.Size([1, 4, 20])  (num_layers, batch, hidden_size) -- just ONE state, like plain nn.RNN

GRU's return signature matches nn.RNN's EXACTLY (a single h_n, not a tuple) --
this is precisely BECAUSE GRU has no separate cell state to track.


### Line-by-Line Explanation
- Note the return signature `output, h_n` — **identical in structure** to plain `nn.RNN`'s return, genuinely different from `nn.LSTM`'s `output, (h_n, c_n)` tuple — this API consistency directly reflects GRU's architectural simplification (no separate cell state), and is a small but real practical convenience: code written for `nn.RNN` often needs minimal changes to swap in `nn.GRU`, while swapping to `nn.LSTM` requires updating the unpacking logic.

## 4. Choosing Between RNN, LSTM, and GRU in Practice


In [ ]:
print("""
PRACTICAL GUIDANCE (echoing this course's standing "good for learning vs good for
production" distinction):

  Plain nn.RNN:
    - Rarely used in production for anything beyond very short sequences or as a
      teaching tool (Chapter 44) -- genuinely limited by vanishing gradients through
      time for most real tasks.

  nn.LSTM:
    - The traditional, well-established choice for sequence tasks with genuinely
      long-range dependencies, when computational cost is not the primary constraint.

  nn.GRU:
    - A strong, often-preferred default for MANY practical tasks -- fewer parameters,
      faster training/inference, frequently comparable accuracy to LSTM, especially
      valuable when data is limited (fewer parameters -> less overfitting risk,
      directly the Scikit-learn course's bias-variance reasoning, Ch 62 there) or
      when INFERENCE LATENCY matters (Ch 71).

  IN MODERN PRACTICE (2020s onward):
    - For NLP specifically, attention-based Transformers (Ch 49-50) have LARGELY
      superseded RNN/LSTM/GRU for most tasks where sufficient compute/data is
      available -- but RNN-family architectures REMAIN genuinely relevant for:
      streaming/online processing (processing one time step at a time, in real
      time, without needing the WHOLE sequence upfront the way standard Transformer
      attention does), resource-constrained deployment, and some time-series
      forecasting tasks (Chapter 67, 91) where Transformers don't always
      provide a clear advantage over well-tuned recurrent approaches.
""")



PRACTICAL GUIDANCE (echoing this course's standing "good for learning vs good for
production" distinction):

  Plain nn.RNN:
    - Rarely used in production for anything beyond very short sequences or as a
      teaching tool (Chapter 44) -- genuinely limited by vanishing gradients through
      time for most real tasks.

  nn.LSTM:
    - The traditional, well-established choice for sequence tasks with genuinely
      long-range dependencies, when computational cost is not the primary constraint.

  nn.GRU:
    - A strong, often-preferred default for MANY practical tasks -- fewer parameters,
      faster training/inference, frequently comparable accuracy to LSTM, especially
      valuable when data is limited (fewer parameters -> less overfitting risk,
      directly the Scikit-learn course's bias-variance reasoning, Ch 62 there) or
      when INFERENCE LATENCY matters (Ch 71).

  IN MODERN PRACTICE (2020s onward):
    - For NLP specifically, attention-based Transformers (Ch 49-50) ha

### Common Mistakes
- Assuming GRU always trains faster and performs identically to LSTM regardless of task — while a reasonable *default* starting assumption, some tasks with especially long-range dependencies genuinely benefit from LSTM's extra gating capacity; the right choice requires empirical comparison on your specific task, not a universal rule.
- Forgetting GRU's simpler return signature when converting code from LSTM to GRU — a straightforward, common conversion mistake.
- Reflexively choosing an RNN-family architecture for a new NLP task in 2024+ without considering whether a Transformer-based approach (Chapters 49-53) would be more effective — worth being aware of the current state of the field, not just this course's historical architecture progression.

### Interview Insight
**Q: When might you choose GRU over LSTM for a new sequence modeling task?**
A: GRU is a reasonable default choice when training/inference speed matters (GRU has fewer parameters and less computation per time step than LSTM), when the training dataset is relatively limited (fewer parameters mean lower overfitting risk, directly the same bias-variance reasoning that applies to choosing a simpler classical model on limited data), or as a first, cheaper baseline to try before committing to LSTM's added complexity. LSTM remains the better choice when the task genuinely requires modeling very long-range dependencies and sufficient data/compute is available to support its larger parameter count — the right choice ultimately requires empirical comparison on the specific task rather than a universal rule, exactly the Chapter 52 (Scikit-learn course) fair-comparison discipline applied here.

### Exercise
Build both an `nn.LSTM` and an `nn.GRU` with matching `input_size`/`hidden_size`, run the same input through both, and compare their parameter counts directly (`sum(p.numel() for p in ...parameters())`).

### Quiz (Chapter 46)
1. What's the key structural simplification GRU makes relative to LSTM?
2. Why does GRU's return signature match `nn.RNN`'s rather than `nn.LSTM`'s?
3. In what scenario might GRU be preferred over LSTM, and vice versa?
4. What has "largely superseded" RNN-family architectures for many modern NLP tasks, and where do RNN-family architectures remain genuinely relevant?


---
# CHAPTER 47 — NLP WITH PYTORCH

## 1. Concept
Before any neural network can process text, raw strings must become numbers — this chapter builds the complete text-preprocessing pipeline: vocabulary construction, tokenization, converting tokens to integer IDs, padding, and attention masks — the standard foundation underlying every NLP architecture in this course from here forward.

## 2. Tokenization and Vocabulary Construction


In [ ]:
from collections import Counter

sentences = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "cats and dogs are pets",
]

# STEP 1: TOKENIZE -- split raw text into individual tokens (here, simple whitespace
# splitting; real pipelines typically use a more sophisticated tokenizer, Chapter 53)
tokenized = [sentence.lower().split() for sentence in sentences]
print("Tokenized:", tokenized)

# STEP 2: BUILD VOCABULARY -- collect every UNIQUE token, assign each a unique integer ID
all_tokens = [token for sentence in tokenized for token in sentence]
token_counts = Counter(all_tokens)
print("\nToken frequencies:", token_counts)

# Special tokens are STANDARD, essential additions: <PAD> for padding, <UNK> for
# genuinely unknown/out-of-vocabulary words encountered later (e.g., at inference time)
vocab = {"<PAD>": 0, "<UNK>": 1}
for token in sorted(token_counts.keys()):
    vocab[token] = len(vocab)

print("\nVocabulary (token -> ID):", vocab)
print("Vocabulary size:", len(vocab))


Tokenized: [['the', 'cat', 'sat', 'on', 'the', 'mat'], ['the', 'dog', 'ran', 'in', 'the', 'park'], ['cats', 'and', 'dogs', 'are', 'pets']]

Token frequencies: Counter({'the': 4, 'cat': 1, 'sat': 1, 'on': 1, 'mat': 1, 'dog': 1, 'ran': 1, 'in': 1, 'park': 1, 'cats': 1, 'and': 1, 'dogs': 1, 'are': 1, 'pets': 1})

Vocabulary (token -> ID): {'<PAD>': 0, '<UNK>': 1, 'and': 2, 'are': 3, 'cat': 4, 'cats': 5, 'dog': 6, 'dogs': 7, 'in': 8, 'mat': 9, 'on': 10, 'park': 11, 'pets': 12, 'ran': 13, 'sat': 14, 'the': 15}
Vocabulary size: 16


### Line-by-Line Explanation
- `<PAD>` (conventionally ID `0`) — the padding token, used to fill shorter sequences up to a common length within a batch (Chapter 26's `pad_sequence`) — a model must learn to essentially ignore this token's contribution (or be explicitly told to via an attention mask, next).
- `<UNK>` — the "unknown" token, essential for handling words at **inference time** that never appeared in the training vocabulary at all — without it, encountering a genuinely new word would have no valid integer ID to map to, causing an error; `<UNK>` provides a defined fallback.

## 3. Converting Text to Token IDs, and Back


In [ ]:
def encode_sentence(sentence: str, vocab: dict) -> list:
    tokens = sentence.lower().split()
    return [vocab.get(token, vocab["<UNK>"]) for token in tokens]   # .get() with a DEFAULT handles unknown words


def decode_ids(ids: list, vocab: dict) -> str:
    id_to_token = {v: k for k, v in vocab.items()}   # build the REVERSE mapping
    return " ".join(id_to_token[i] for i in ids)


encoded = encode_sentence("the cat sat on the mat", vocab)
print("Encoded:", encoded)

decoded = decode_ids(encoded, vocab)
print("Decoded back:", decoded)

# Demonstrating <UNK> handling for a genuinely NEW word never seen during vocabulary construction
encoded_with_unknown = encode_sentence("the elephant sat quietly", vocab)
print("\nEncoded sentence with unseen words ('elephant', 'quietly'):", encoded_with_unknown)
print("Decoded:", decode_ids(encoded_with_unknown, vocab), " (unknown words correctly mapped to <UNK>)")


Encoded: [15, 4, 14, 10, 15, 9]
Decoded back: the cat sat on the mat

Encoded sentence with unseen words ('elephant', 'quietly'): [15, 1, 14, 1]
Decoded: the <UNK> sat <UNK>  (unknown words correctly mapped to <UNK>)


## 4. Padding and Attention Masks — Batching Variable-Length Text


In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

encoded_sentences = [torch.tensor(encode_sentence(s, vocab)) for s in sentences]
print("Individual sequence lengths:", [len(s) for s in encoded_sentences])

padded_batch = pad_sequence(encoded_sentences, batch_first=True, padding_value=vocab["<PAD>"])
print("\nPadded batch shape:", padded_batch.shape)
print(padded_batch)

# ATTENTION MASK: a boolean/binary tensor marking which positions are REAL tokens (1)
# vs PADDING (0) -- essential so the model can be told to IGNORE padded positions
attention_mask = (padded_batch != vocab["<PAD>"]).long()
print("\nAttention mask (1=real token, 0=padding):")
print(attention_mask)


Individual sequence lengths: [6, 6, 5]

Padded batch shape: torch.Size([3, 6])
tensor([[15,  4, 14, 10, 15,  9],
        [15,  6, 13,  8, 15, 11],
        [ 5,  2,  7,  3, 12,  0]])

Attention mask (1=real token, 0=padding):
tensor([[1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0]])


### Why the Attention Mask Matters
Directly extending Chapter 26's lesson: the padded `<PAD>` tokens have a *valid* integer ID (`0`) and would be processed by the model's embedding layer (Chapter 48) exactly like any other token unless explicitly told otherwise — the attention mask is precisely this "told otherwise" mechanism, used inside attention computations (Chapter 49) to force the model to assign essentially zero attention weight to padded positions, and inside loss computations to exclude padded positions from contributing to the training signal at all.

### Common Mistakes
- Forgetting `<UNK>`/`<PAD>` special tokens entirely — causes crashes on genuinely new vocabulary at inference time, or incorrect batching of variable-length sequences.
- Building the vocabulary from the **entire** dataset (train + validation + test combined) rather than training data only — a genuine, if often small, form of data leakage (the Scikit-learn course's Chapter 56 leakage taxonomy, extended to vocabulary construction specifically).
- Not tracking/using an attention mask at all, and letting the model process padding tokens as if they were genuine content — degrades model quality in a way that's easy to overlook, since it doesn't cause an outright error.
- Using inconsistent tokenization between training and inference (e.g., different lowercasing/punctuation handling) — silently produces different token sequences for what should be identical text, corrupting model behavior at inference time.

### Interview Insight
**Q: Why is building your vocabulary using only the training set (not validation/test data too) important, and what specific failure does this prevent?**
A: This directly parallels the general data leakage principle (Scikit-learn course Chapter 56) — if the vocabulary is built using validation/test data, the model implicitly "knows about" every word that will appear in evaluation, which wouldn't be true for genuinely new, real-world text at deployment time. Building the vocabulary from training data alone, and handling any genuinely new words encountered later via the `<UNK>` token, ensures the evaluation setup honestly reflects how the model would behave on truly unseen text in production — including test-set words the model has never encountered, which is a realistic and expected scenario the `<UNK>` mechanism is specifically designed to handle gracefully.

### Exercise
1. Build a vocabulary from a small set of your own sentences, correctly including `<PAD>` and `<UNK>`.
2. Encode a batch of your sentences, pad them, and construct the corresponding attention mask.

### Quiz (Chapter 47)
1. What do `<PAD>` and `<UNK>` each specifically solve?
2. Why must the vocabulary be built from training data only?
3. What does an attention mask communicate to the model, and why is it necessary even though padding tokens have a valid integer ID?


---
# CHAPTER 48 — EMBEDDINGS

## 1. Concept
`nn.Embedding` converts sparse, discrete token IDs into dense, continuous vectors — the standard first layer of virtually every NLP model. Unlike one-hot encoding (Scikit-learn course Chapter 6 — a sparse, high-dimensional, purely categorical representation with no notion of similarity between categories), embeddings are **learned**, dense vectors where semantically similar tokens end up with **similar** vector representations, purely as an emergent consequence of training.

## 2. `nn.Embedding` — Mechanics


In [ ]:
import torch
import torch.nn as nn

vocab_size = 1000
embed_dim = 64

embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)

token_ids = torch.tensor([[5, 20, 999, 0], [3, 3, 15, 0]])   # (batch=2, seq_len=4) -- integer IDs
embedded = embedding(token_ids)

print("Input (token IDs) shape:", token_ids.shape)
print("Output (embeddings) shape:", embedded.shape, " -- ONE dense vector PER TOKEN")
print("\nembedding.weight.shape:", embedding.weight.shape, " (vocab_size, embed_dim)")
print("\nnn.Embedding is FUNDAMENTALLY just a LOOKUP TABLE: token ID 'i' retrieves ROW 'i'")
print("of the weight matrix -- and that weight matrix is LEARNED via ordinary backpropagation,")
print("exactly like any other layer's weights (Chapter 13-14).")


Input (token IDs) shape: torch.Size([2, 4])
Output (embeddings) shape: torch.Size([2, 4, 64])  -- ONE dense vector PER TOKEN

embedding.weight.shape: torch.Size([1000, 64])  (vocab_size, embed_dim)

nn.Embedding is FUNDAMENTALLY just a LOOKUP TABLE: token ID 'i' retrieves ROW 'i'
of the weight matrix -- and that weight matrix is LEARNED via ordinary backpropagation,
exactly like any other layer's weights (Chapter 13-14).


### Line-by-Line Explanation
- `nn.Embedding(num_embeddings, embedding_dim)` — internally holds a single weight matrix of shape `(num_embeddings, embedding_dim)`; calling `embedding(token_ids)` is literally an indexing/lookup operation, retrieving row `token_ids[i]` for each position — genuinely one of the simplest layers in PyTorch, mechanically, despite its conceptual importance.
- The weight matrix starts **randomly initialized** (Chapter 30) and is updated via ordinary backpropagation during training, just like any other parameter — there's no special "embedding training algorithm"; the network simply learns, through gradient descent on whatever downstream task it's trained for, that certain token vectors should end up close together in the embedding space because doing so reduces the loss.

## 3. Why Embeddings Capture Semantic Similarity — the Emergent Property


In [ ]:
print("""
WHY do semantically similar words end up with similar embedding vectors, given that
NOTHING in nn.Embedding's mechanics explicitly enforces this?

Consider training a next-word-prediction model (Chapter 54's language modeling task)
on a large text corpus. The words "cat" and "dog" appear in genuinely SIMILAR
CONTEXTS across many sentences ("I have a pet ___", "The ___ is sleeping",
"My ___ needs food"). For the model to correctly predict what comes NEXT after
either word with LOW LOSS, it's advantageous for their EMBEDDING VECTORS to
respond SIMILARLY to the surrounding context -- and gradient descent, purely by
minimizing the training loss, discovers and reinforces this similarity over many
training steps, WITHOUT ever being explicitly told "cat and dog are similar."

This emergent property is EXACTLY what famous results like:
    embedding("king") - embedding("man") + embedding("woman") ~= embedding("queen")
are demonstrating -- learned embedding spaces capture genuine SEMANTIC and even
ANALOGICAL relationships, purely as a byproduct of optimizing a predictive task.
""")



WHY do semantically similar words end up with similar embedding vectors, given that
NOTHING in nn.Embedding's mechanics explicitly enforces this?

Consider training a next-word-prediction model (Chapter 54's language modeling task)
on a large text corpus. The words "cat" and "dog" appear in genuinely SIMILAR
CONTEXTS across many sentences ("I have a pet ___", "The ___ is sleeping",
"My ___ needs food"). For the model to correctly predict what comes NEXT after
either word with LOW LOSS, it's advantageous for their EMBEDDING VECTORS to
respond SIMILARLY to the surrounding context -- and gradient descent, purely by
minimizing the training loss, discovers and reinforces this similarity over many
training steps, WITHOUT ever being explicitly told "cat and dog are similar."

This emergent property is EXACTLY what famous results like:
    embedding("king") - embedding("man") + embedding("woman") ~= embedding("queen")
are demonstrating -- learned embedding spaces capture genuine SEMANTIC and 

## 4. Using Pretrained Embeddings vs. Training From Scratch


In [ ]:
print("""
Exactly analogous to Chapter 38's TRANSFER LEARNING for computer vision, you can
INITIALIZE an nn.Embedding layer's weights with PRETRAINED word vectors (historically
Word2Vec or GloVe; in modern practice, more often the embeddings from a pretrained
Transformer like BERT, Chapter 53) rather than random initialization -- giving the
model a genuinely useful semantic starting point before ANY task-specific training
has even happened.

WHEN THIS MATTERS MOST: small training datasets, where learning good embeddings
FROM SCRATCH (purely from your own limited data) would be difficult -- pretrained
embeddings inject broad, general linguistic knowledge learned from a vastly larger
corpus than your specific task's dataset likely contains.

MODERN PRACTICE: rather than loading STANDALONE pretrained embeddings (Word2Vec/
GloVe) into an nn.Embedding layer, it's now FAR more common to use a full PRETRAINED
TRANSFORMER model (Chapter 53's Hugging Face ecosystem) whose embeddings are
CONTEXTUAL (the SAME word gets a DIFFERENT vector depending on its surrounding
sentence) -- a genuinely more powerful representation than Word2Vec/GloVe's FIXED,
context-INDEPENDENT vectors, which is exactly WHY the field moved toward this
approach (directly motivating Chapters 49-53's progression).
""")



Exactly analogous to Chapter 38's TRANSFER LEARNING for computer vision, you can
INITIALIZE an nn.Embedding layer's weights with PRETRAINED word vectors (historically
Word2Vec or GloVe; in modern practice, more often the embeddings from a pretrained
Transformer like BERT, Chapter 53) rather than random initialization -- giving the
model a genuinely useful semantic starting point before ANY task-specific training
has even happened.

WHEN THIS MATTERS MOST: small training datasets, where learning good embeddings
FROM SCRATCH (purely from your own limited data) would be difficult -- pretrained
embeddings inject broad, general linguistic knowledge learned from a vastly larger
corpus than your specific task's dataset likely contains.

MODERN PRACTICE: rather than loading STANDALONE pretrained embeddings (Word2Vec/
GloVe) into an nn.Embedding layer, it's now FAR more common to use a full PRETRAINED
TRANSFORMER model (Chapter 53's Hugging Face ecosystem) whose embeddings are
CONTEXTUAL (the 

### Common Mistakes
- Confusing `nn.Embedding` with one-hot encoding conceptually — one-hot vectors are sparse, fixed, and have no learned notion of similarity between categories (Scikit-learn course Chapter 6's rule against implying false ordinal relationships applies conceptually here too, in reverse: one-hot deliberately encodes NO relationship, while embeddings deliberately learn genuine relationships); embeddings are dense, learned, and specifically designed to capture similarity.
- Choosing an embedding dimension arbitrarily without considering the tradeoff — larger dimensions can capture more nuance but require more data/parameters to learn well and risk overfitting on smaller datasets; common practical values range from tens to a few hundred dimensions for smaller custom vocabularies, with modern large language models often using far larger dimensions (Chapter 54).
- Using a fixed, pretrained (non-contextual) embedding for a task where word meaning genuinely depends on context (e.g., "bank" as a financial institution vs. a riverbank) — this is exactly the specific limitation that motivated the field's shift toward contextual embeddings (BERT-style, Chapter 53).

### Interview Insight
**Q: How does `nn.Embedding` differ fundamentally from one-hot encoding, and why does this distinction matter for a model's ability to generalize?**
A: One-hot encoding represents each category as a sparse vector with a single `1` and all other positions `0` — every category is equally "far" from every other category by construction, with no learned or inherent notion of similarity. `nn.Embedding` instead represents each token as a dense, low(er)-dimensional vector whose values are learned parameters, updated via ordinary backpropagation on whatever downstream task the model is trained on — through this training process, tokens that behave similarly in context tend to develop similar embedding vectors, purely as an emergent consequence of the model minimizing its loss. This matters for generalization because a model working with meaningful embedding similarity can transfer learned behavior to related-but-not-identical tokens more gracefully than a model using one-hot vectors, which treat every category as completely, arbitrarily distinct with no shared structure at all.

### Exercise
1. Build an `nn.Embedding(vocab_size=500, embedding_dim=32)`, look up embeddings for a batch of token IDs, and confirm the output shape matches your prediction.
2. Explain, in your own words (no code needed), why the embedding weight matrix genuinely being learned (not fixed) is what allows semantic similarity to emerge during training.

### Quiz (Chapter 48)
1. Mechanically, what does `nn.Embedding(token_ids)` actually compute?
2. Why do semantically similar words end up with similar embedding vectors, despite nothing explicitly enforcing this?
3. What's the key difference between "contextual" embeddings (BERT-style) and "fixed" embeddings (Word2Vec/GloVe-style), and why does this matter for words with multiple meanings?
4. When is initializing with pretrained embeddings most valuable, versus training an embedding layer entirely from scratch?


---
# CHAPTER 49 — ATTENTION

## 1. Concept
Attention lets a model **dynamically weight** how much each element of a sequence should influence the representation of every other element — directly addressing RNN/LSTM's core limitation (Chapter 44-45): a recurrent hidden state must compress the *entire* preceding sequence into one fixed-size vector, inevitably losing information for long sequences. Attention instead lets every position **directly** access every other position, with learned weights determining relevance — no compression bottleneck, no vanishing-gradient-through-time problem at all (since there's no recurrence).

## 2. Query, Key, Value — the Core Vocabulary
Think of attention as a **soft, differentiable lookup**:
- **Query (Q)**: "what am I looking for?" — one query vector per position that's currently being computed.
- **Key (K)**: "what do I contain?" — one key vector per position being attended TO (potentially every position in the sequence, including the query position itself).
- **Value (V)**: "what information do I actually provide, if selected?" — the actual content retrieved, weighted by how well the query matched each key.

## 3. Scaled Dot-Product Attention, Implemented


In [ ]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(query, key, value, mask=None):
    """query, key, value shapes: (batch, seq_len, d_k)
    Returns: (output, attention_weights)"""
    d_k = query.size(-1)

    # STEP 1: compute raw similarity scores between EVERY query and EVERY key --
    # exactly the einsum pattern from Chapter 10: 'bqd,bkd->bqk'
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)   # (batch, seq_len_q, seq_len_k)

    # STEP 2: apply the mask (Chapter 47) -- setting masked positions to a LARGE
    # NEGATIVE value so softmax essentially assigns them ZERO probability
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # STEP 3: softmax converts raw scores into a PROBABILITY DISTRIBUTION over
    # key positions (Scikit-learn course's softmax concept, Ch 15 there, in a new context)
    attention_weights = F.softmax(scores, dim=-1)

    # STEP 4: use the attention weights to compute a WEIGHTED AVERAGE of the VALUES
    output = torch.matmul(attention_weights, value)   # (batch, seq_len_q, d_k)

    return output, attention_weights


batch, seq_len, d_k = 2, 5, 16
q = torch.randn(batch, seq_len, d_k)
k = torch.randn(batch, seq_len, d_k)
v = torch.randn(batch, seq_len, d_k)

output, weights = scaled_dot_product_attention(q, k, v)
print("Output shape:", output.shape, " (same as query/value's shape)")
print("Attention weights shape:", weights.shape, " (batch, seq_len_q, seq_len_k)")
print("\nEach ROW of attention_weights sums to 1.0 (it's a softmax-normalized")
print("probability distribution over WHICH key positions to attend to):")
print(weights[0].sum(dim=-1))


Output shape: torch.Size([2, 5, 16])  (same as query/value's shape)
Attention weights shape: torch.Size([2, 5, 5])  (batch, seq_len_q, seq_len_k)

Each ROW of attention_weights sums to 1.0 (it's a softmax-normalized
probability distribution over WHICH key positions to attend to):
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


### Line-by-Line Explanation — Why Each Step Exists
- `torch.matmul(query, key.transpose(-2, -1))` — computes the dot product between every query and every key, producing a `(seq_len_q, seq_len_k)` similarity matrix per batch element; a higher dot product means the query and key vectors point in a more similar direction, interpreted as "this key is more relevant to this query."
- `/ math.sqrt(d_k)` — **the scaling in "scaled" dot-product attention**: without this, as `d_k` (the vector dimension) grows large, dot products can become very large in magnitude purely as a statistical consequence of summing more terms — pushing softmax into an extremely "peaked," saturated regime (Chapter 18's saturation concept, now applied to attention weights) where gradients become vanishingly small. Dividing by `sqrt(d_k)` keeps the scores in a numerically well-behaved range regardless of dimension, directly preserving healthy gradient flow through the softmax.
- `masked_fill(mask == 0, float("-inf"))` — setting masked (padding) positions' scores to negative infinity **before** softmax ensures they receive essentially exactly `0` probability after softmax (since `exp(-inf) = 0`) — this is precisely how Chapter 47's attention mask actually gets used mechanically inside the attention computation.
- The final `torch.matmul(attention_weights, value)` computes a **weighted average** of all value vectors, where the weights are exactly the softmax-normalized attention scores — positions the model "attends to" more strongly contribute proportionally more to the output.

## 4. Multi-Head Attention — Running Several Attention Computations in Parallel


In [ ]:
import torch.nn as nn

multihead_attn = nn.MultiheadAttention(embed_dim=64, num_heads=8, batch_first=True)

x = torch.randn(2, 10, 64)   # (batch, seq_len, embed_dim)

# Self-attention: query, key, AND value all come from the SAME sequence
output, attention_weights = multihead_attn(query=x, key=x, value=x)
print("Output shape:", output.shape, " (SAME as input -- self-attention PRESERVES shape)")
print("Attention weights shape:", attention_weights.shape, " (batch, seq_len_q, seq_len_k, AVERAGED across heads)")

print("""
WHY MULTIPLE HEADS? Each of the 8 heads learns to attend to DIFFERENT kinds of
relationships INDEPENDENTLY (one head might learn to track subject-verb
agreement, another might track nearby words, another long-range topical
relevance) -- their outputs are CONCATENATED and linearly projected back to
embed_dim, giving the model SEVERAL independent "attention perspectives"
on the same sequence simultaneously, rather than being forced to compress
everything into just ONE attention pattern.
""")


Output shape: torch.Size([2, 10, 64])  (SAME as input -- self-attention PRESERVES shape)
Attention weights shape: torch.Size([2, 10, 10])  (batch, seq_len_q, seq_len_k, AVERAGED across heads)

WHY MULTIPLE HEADS? Each of the 8 heads learns to attend to DIFFERENT kinds of
relationships INDEPENDENTLY (one head might learn to track subject-verb
agreement, another might track nearby words, another long-range topical
relevance) -- their outputs are CONCATENATED and linearly projected back to
embed_dim, giving the model SEVERAL independent "attention perspectives"
on the same sequence simultaneously, rather than being forced to compress
everything into just ONE attention pattern.



### Common Mistakes
- Forgetting the `sqrt(d_k)` scaling factor — can cause softmax saturation and poor gradient flow, especially with larger embedding dimensions, a subtle training-quality issue rather than an outright crash.
- Applying the attention mask incorrectly (e.g., masking BEFORE the scaling, or using `0` instead of `-inf` for masked positions before softmax) — produces incorrect, non-zero attention weight leaking to padding positions.
- Confusing self-attention (query, key, value all from the same sequence — used in Transformer encoders) with cross-attention (query from one sequence, key/value from a different sequence — used in Transformer decoders attending to encoder output, Chapter 50).

### Interview Insight
**Q: Why does scaled dot-product attention divide by `sqrt(d_k)` before applying softmax?**
A: As the query/key vector dimension `d_k` grows, the dot product between two random vectors tends to grow in magnitude too (it's a sum of `d_k` terms) — without correction, this pushes the raw attention scores into a range where softmax becomes extremely "peaked" (saturating toward a near-one-hot distribution), which in turn causes the softmax function's gradient to become very small (an analogous saturation problem to Chapter 18's sigmoid/tanh saturation), hurting gradient flow during training. Dividing by `sqrt(d_k)` rescales the dot products to keep their variance roughly constant regardless of dimension, keeping softmax's output — and its gradient — in a numerically healthy range.

### Exercise
1. Implement `scaled_dot_product_attention` from memory (or close to it), and verify the output attention weights sum to 1 along the correct dimension.
2. Build an `nn.MultiheadAttention` layer and run self-attention on a sample sequence, verifying output shape matches input shape.

### Quiz (Chapter 49)
1. Explain Query, Key, and Value in your own words, using the "soft lookup" analogy.
2. Why does scaled dot-product attention divide by `sqrt(d_k)`?
3. How does the attention mask mechanically prevent attending to padding positions?
4. Why does multi-head attention use multiple heads rather than one larger single-head attention computation?


---
# CHAPTER 50 — TRANSFORMERS

## 1. Concept
The Transformer architecture (2017, "Attention Is All You Need") assembles Chapter 49's attention mechanism, together with several supporting components, into a complete architecture that processes an **entire sequence in parallel** — no recurrence at all — while still modeling relationships between every pair of positions via attention. This parallelism (versus RNN's strictly sequential, one-time-step-at-a-time processing) is a major reason Transformers train dramatically faster on modern parallel hardware (GPUs), on top of generally producing stronger results for most NLP tasks.

## 2. The Full Architecture, Piece by Piece
```
Input tokens
     ↓
Token Embeddings (Chapter 48) + Positional Encoding (this chapter -- REPLACES recurrence's implicit ordering)
     ↓
[ Transformer Block ] x N:
     Multi-Head Self-Attention (Chapter 49)
          ↓
     Residual connection + LayerNorm (Chapter 17, 39's residual concept)
          ↓
     Feed-Forward Network (two Linear layers with an activation between them)
          ↓
     Residual connection + LayerNorm
     ↓
Output (task-specific head: classification, next-token prediction, etc.)
```

## 3. Positional Encoding — Why It's Needed at All


In [ ]:
import torch
import math

def positional_encoding(seq_len, d_model):
    """The ORIGINAL sinusoidal positional encoding scheme -- a FIXED (not learned)
    pattern giving each position a UNIQUE, smoothly-varying signature."""
    position = torch.arange(seq_len).unsqueeze(1).float()          # (seq_len, 1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)   # even dimensions: sine
    pe[:, 1::2] = torch.cos(position * div_term)   # odd dimensions: cosine
    return pe


pe = positional_encoding(seq_len=50, d_model=64)
print("Positional encoding shape:", pe.shape)

print("""
WHY IS THIS NEEDED? Unlike an RNN, which processes tokens ONE AT A TIME IN ORDER
(making sequence position implicit in WHEN each token is processed), a
Transformer's self-attention treats its input as an UNORDERED SET of positions --
attention itself has NO inherent notion of "position 3 comes before position 5."

WITHOUT positional encoding, shuffling a sentence's WORD ORDER would produce
IDENTICAL attention computations (just with the outputs correspondingly
reordered) -- the model would have NO way to distinguish "the dog bit the man"
from "the man bit the dog" purely from attention's own mechanics.

Positional encoding ADDS a unique, position-dependent signal directly to each
token's embedding BEFORE it enters the attention layers -- giving the model the
information it needs to recover and use word ORDER, despite attention's own
inherently order-agnostic computation.
""")


Positional encoding shape: torch.Size([50, 64])

WHY IS THIS NEEDED? Unlike an RNN, which processes tokens ONE AT A TIME IN ORDER
(making sequence position implicit in WHEN each token is processed), a
Transformer's self-attention treats its input as an UNORDERED SET of positions --
attention itself has NO inherent notion of "position 3 comes before position 5."

WITHOUT positional encoding, shuffling a sentence's WORD ORDER would produce
IDENTICAL attention computations (just with the outputs correspondingly
reordered) -- the model would have NO way to distinguish "the dog bit the man"
from "the man bit the dog" purely from attention's own mechanics.

Positional encoding ADDS a unique, position-dependent signal directly to each
token's embedding BEFORE it enters the attention layers -- giving the model the
information it needs to recover and use word ORDER, despite attention's own
inherently order-agnostic computation.



### Modern Practice: Learned vs. Fixed Positional Encoding
The sinusoidal scheme above is the **original** Transformer paper's choice, fully fixed (not learned) — a genuinely elegant mathematical construction. Many modern architectures instead use a **learned** positional embedding (functionally an `nn.Embedding` indexed by position rather than token identity) — simpler to implement and often performs comparably or better; some very recent architectures use still other schemes (like RoPE — Rotary Position Embedding, common in modern LLMs). The specific scheme matters less than understanding **why** positional information must be injected at all, which is this section's core lesson.

## 4. The Transformer Block — Residual Connections and LayerNorm


In [ ]:
import torch.nn as nn

class TransformerBlock(nn.Module):
    """A single Transformer encoder block: self-attention + feed-forward,
    EACH wrapped in a residual connection + LayerNorm -- directly reusing
    Chapter 39's residual-connection insight (gradient flow) and Chapter 17's
    LayerNorm (batch-independence, essential for variable-length sequences)."""

    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, dropout=dropout)
        self.norm1 = nn.LayerNorm(embed_dim)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),                        # Chapter 18's lesson: GELU is the modern Transformer default
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention sub-block, with a RESIDUAL connection
        attn_output, _ = self.self_attn(x, x, x, attn_mask=mask)
        x = self.norm1(x + self.dropout(attn_output))   # residual ADD, then normalize

        # Feed-forward sub-block, ALSO with a residual connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))       # residual ADD, then normalize

        return x


block = TransformerBlock(embed_dim=64, num_heads=8, ff_dim=256)
x = torch.randn(2, 10, 64)   # (batch, seq_len, embed_dim)
output = block(x)
print("Input shape:", x.shape, " Output shape:", output.shape, " (IDENTICAL -- blocks preserve shape,")
print("allowing them to be STACKED N times in sequence, Chapter 51)")


Input shape: torch.Size([2, 10, 64])  Output shape: torch.Size([2, 10, 64])  (IDENTICAL -- blocks preserve shape,
allowing them to be STACKED N times in sequence, Chapter 51)


### Line-by-Line Explanation — Why Every Piece Is There
- `x + self.dropout(attn_output)` — the residual connection (Chapter 39's insight, directly reused): even if the attention sub-block's gradient contribution were to shrink through many stacked blocks, the residual path preserves a full-strength gradient signal, exactly the mechanism that lets Transformers stack dozens of blocks successfully (modern LLMs stack far more than the original paper's 6, Chapter 54).
- `self.norm1(...)` — **LayerNorm** (Chapter 17), applied **after** the residual addition — normalizes each token's representation independently of batch composition and sequence length, exactly the property that made it the standard choice for sequence models over BatchNorm.
- `nn.GELU()` in the feed-forward block — directly Chapter 18's lesson: GELU is the standard modern Transformer activation choice, smoother than ReLU with no hard "dead neuron" zone.
- The feed-forward network's expansion (`embed_dim → ff_dim → embed_dim`, typically `ff_dim` is 4x `embed_dim`) gives the block additional per-position representational capacity beyond what attention alone provides — attention handles *mixing information across positions*; the feed-forward network processes *each position independently*, adding depth and non-linearity.

### Common Mistakes
- Forgetting positional encoding entirely — the model would be fundamentally unable to use word order information, a severe (though not error-raising) correctness bug.
- Applying LayerNorm **before** the residual addition instead of after (a genuine architectural variant — "Pre-LN" vs. "Post-LN" — with real training-stability tradeoffs) without being deliberate about which convention is being used; both exist in real architectures, but mixing them up relative to a specific paper/codebase's intended design is a common source of subtle bugs when reimplementing published work (Chapter 104).
- Not stacking enough Transformer blocks for the task's complexity, or stacking far more than the (limited) available training data can support — depth is a genuine capacity/data tradeoff, exactly like any other model complexity choice (Scikit-learn course's bias-variance framework).

### Interview Insight
**Q: Why does a Transformer need explicit positional encoding, when an RNN doesn't need anything analogous?**
A: An RNN processes tokens strictly one at a time, in sequence order — the token's position is implicitly encoded by *when* it's processed relative to the others, since the hidden state genuinely accumulates information in that fixed temporal order. A Transformer's self-attention, by contrast, computes relationships between all pairs of positions **simultaneously and in parallel**, with no inherent sense of "before" or "after" built into the attention computation itself — mathematically, shuffling the input sequence's order would produce the same set of attention computations, just correspondingly reordered, giving the model no way to distinguish sequences that differ only in word order. Positional encoding solves this by injecting an explicit, position-dependent signal into each token's representation before it enters the attention layers, giving the otherwise order-agnostic attention mechanism the information it needs to actually use sequence order.

### Exercise
1. Implement the sinusoidal positional encoding function from memory (or close to it) and visualize a few dimensions' values across positions using Matplotlib.
2. Build a `TransformerBlock` and stack 3 of them sequentially, verifying the final output shape matches the input shape.

### Quiz (Chapter 50)
1. Why can a Transformer process an entire sequence in parallel, while an RNN cannot?
2. What specific problem does positional encoding solve?
3. What TWO residual connections exist within a single Transformer block, and what does each protect?
4. Why is LayerNorm (not BatchNorm) the standard normalization choice inside a Transformer block?


---
# CHAPTER 51 — IMPLEMENT A TRANSFORMER FROM SCRATCH

## 1. Concept
This chapter assembles every piece from Chapters 48-50 into one complete, working (simplified, educational) Transformer encoder — not calling `nn.Transformer` directly, but building the full architecture explicitly, to cement genuine understanding of how every piece fits together.

## 2. The Complete Assembly


In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    """Wraps Chapter 50's sinusoidal positional encoding as a proper nn.Module,
    using register_buffer (Chapter 16) since it's FIXED, not a trainable parameter."""

    def __init__(self, embed_dim, max_seq_len=512):
        super().__init__()
        position = torch.arange(max_seq_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe = torch.zeros(max_seq_len, embed_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_seq_len, embed_dim) -- broadcasts over batch

    def forward(self, x):
        # x: (batch, seq_len, embed_dim) -- add the positional encoding, SLICED to the actual seq_len
        return x + self.pe[:, :x.size(1), :]


class TransformerBlock(nn.Module):
    """Identical to Chapter 50's TransformerBlock -- reused here as a component."""
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, dropout=dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff_dim, embed_dim),
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_output, _ = self.self_attn(x, x, x, key_padding_mask=mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x


class SimpleTransformerEncoder(nn.Module):
    """A COMPLETE, minimal Transformer encoder -- from raw token IDs to a
    sequence of contextualized representations."""

    def __init__(self, vocab_size, embed_dim=128, num_heads=8, ff_dim=512, num_layers=4, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_encoding = PositionalEncoding(embed_dim, max_seq_len)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout) for _ in range(num_layers)
        ])   # nn.ModuleList -- like a Python list, but CORRECTLY registers each block's parameters (Ch 16)
        self.dropout = nn.Dropout(dropout)

    def forward(self, token_ids, padding_mask=None):
        x = self.token_embedding(token_ids)          # (batch, seq_len, embed_dim)
        x = self.positional_encoding(x)                # position info ADDED
        x = self.dropout(x)

        for block in self.blocks:                       # STACK of N Transformer blocks, applied in sequence
            x = block(x, mask=padding_mask)

        return x   # (batch, seq_len, embed_dim) -- a CONTEXTUALIZED representation for EVERY token


model = SimpleTransformerEncoder(vocab_size=5000, embed_dim=128, num_heads=8, num_layers=4)
token_ids = torch.randint(0, 5000, (4, 20))   # (batch=4, seq_len=20)
output = model(token_ids)

print("Input shape:", token_ids.shape)
print("Output shape:", output.shape, " (batch, seq_len, embed_dim) -- ONE contextualized vector PER TOKEN")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


Input shape: torch.Size([4, 20])
Output shape: torch.Size([4, 20, 128])  (batch, seq_len, embed_dim) -- ONE contextualized vector PER TOKEN

Total parameters: 1,433,088


### Line-by-Line Explanation — Why Each Design Choice Is There
- `nn.ModuleList` (not a plain Python list) — **essential**, not a style preference: a plain Python `list` of `nn.Module`s would NOT be recognized by PyTorch's parameter-tracking machinery (Chapter 16) — `model.parameters()` would silently miss every parameter inside those blocks, and they would never train. `nn.ModuleList` specifically exists to correctly register a variable-length collection of submodules.
- `register_buffer("pe", ...)` for the positional encoding — since it's a **fixed** (not learned) mathematical pattern, it must never appear in `model.parameters()` (the optimizer should never update it) but it does need to move correctly with `.to(device)` and appear in `.state_dict()` for consistency — exactly Chapter 16's buffer use case.
- The output is a full sequence of contextualized representations `(batch, seq_len, embed_dim)` — one vector per input token, each having "attended to" every other token through the stacked attention layers — this is the **encoder's** raw output; a task-specific head (classification, Chapter 52; language modeling, Chapter 55) is applied on top of this to produce the actual final predictions.

### Common Mistakes
- Using a plain Python `list` instead of `nn.ModuleList` for a variable number of submodules — silently breaks parameter registration, meaning those layers never actually train, without any error message at all (a genuinely dangerous silent failure, directly analogous to the Scikit-learn course's silent `cat()`-instead-of-`stack()` mistake).
- Forgetting to add positional encoding before the Transformer blocks — the model loses all access to word order information.
- Confusing the encoder's raw per-token output with a final prediction — a task-specific head (pooling + classification layer, or a language modeling head, Chapters 52 and 55) is still needed on top.

### Interview Insight
**Q: Why must a variable-length collection of submodules (like a list of Transformer blocks) be stored in `nn.ModuleList` rather than a plain Python list?**
A: `nn.Module`'s parameter-tracking machinery (Chapter 16) relies on its special `__setattr__` behavior, which specifically recognizes `nn.Module` instances (and `nn.ModuleList`/`nn.ModuleDict` containers of them) assigned as attributes, registering their parameters into the parent module's tracked parameter tree. A plain Python `list` containing `nn.Module` instances is just an ordinary attribute as far as `nn.Module.__setattr__` is concerned — it has no special awareness that this particular list happens to contain trainable submodules, so none of those submodules' parameters get registered. The practical, damaging consequence: `model.parameters()` would silently omit every parameter inside the plain-list-stored blocks, the optimizer would never update them, and those layers would remain at their random initialization values throughout training — with no error raised anywhere to indicate anything was wrong.

### Exercise
Build the complete `SimpleTransformerEncoder` from memory (or close to it, referring back as needed), run a forward pass, and verify the total parameter count is reasonable for the given `vocab_size`/`embed_dim`/`num_layers` configuration.

### Quiz (Chapter 51)
1. Why must `nn.ModuleList` be used instead of a plain Python list for a stack of Transformer blocks?
2. Why is positional encoding registered as a buffer rather than a parameter?
3. What does `SimpleTransformerEncoder`'s output actually represent, and what's still needed to turn it into a final task prediction?


---
# CHAPTER 52 — NLP TRANSFORMER PROJECT

## Business Problem
Building a complete sentiment classification system using the Transformer encoder built in Chapter 51 — assembling tokenization (Chapter 47), embeddings (Chapter 48), the full Transformer architecture (Chapters 49-51), and a classification head into one genuine, trainable pipeline.

## The Complete Pipeline


In [ ]:
import torch
import torch.nn as nn

class TransformerClassifier(nn.Module):
    """Wraps the Chapter 51 encoder with a classification head -- the standard
    pattern for using a Transformer encoder for SEQUENCE CLASSIFICATION (as
    opposed to per-token/language-modeling tasks, Chapters 55+)."""

    def __init__(self, vocab_size, embed_dim=128, num_heads=8, ff_dim=512,
                 num_layers=4, num_classes=2, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.encoder = SimpleTransformerEncoder(vocab_size, embed_dim, num_heads, ff_dim, num_layers, max_seq_len, dropout)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, token_ids, padding_mask=None):
        encoded = self.encoder(token_ids, padding_mask)   # (batch, seq_len, embed_dim)

        # POOLING: we need ONE vector per SAMPLE (for classification), but the encoder
        # gives us one vector PER TOKEN -- mean-pooling across the sequence dimension
        # is a simple, standard, effective way to summarize the whole sequence
        pooled = encoded.mean(dim=1)   # (batch, embed_dim) -- Chapter 9's dim/keepdim lesson, applied here

        logits = self.classifier(pooled)   # (batch, num_classes) -- raw logits, Ch 19's rule
        return logits


model = TransformerClassifier(vocab_size=5000, embed_dim=128, num_heads=8, num_layers=4, num_classes=2)
token_ids = torch.randint(0, 5000, (8, 30))   # batch=8, seq_len=30
logits = model(token_ids)
print("Input shape:", token_ids.shape)
print("Output (logits) shape:", logits.shape)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


Input shape: torch.Size([8, 30])
Output (logits) shape: torch.Size([8, 2])

Total parameters: 1,433,346


### Line-by-Line Explanation — Pooling Strategies
- `encoded.mean(dim=1)` — averages across the **sequence** dimension (`dim=1`, since shape is `(batch, seq_len, embed_dim)`), collapsing the per-token representations into one summary vector per sample. This is **mean pooling** — simple and often surprisingly effective, but it treats every token equally regardless of relevance.
- **Alternative pooling strategies** worth knowing: (1) using only the **final** token's representation (`encoded[:, -1, :]`) — works if the architecture ensures the final position has attended to and aggregated the full sequence's context; (2) a dedicated **`[CLS]` token** approach (BERT's method, Chapter 53) — prepending a special learnable token to every input sequence specifically designed to aggregate whole-sequence information via attention, then using only that position's final representation for classification — generally considered more principled than raw mean pooling, since the model can *learn* what to aggregate rather than being forced into a fixed uniform average.

## 3. The Full Training Setup


In [ ]:
from torch.utils.data import Dataset, DataLoader

class SentimentDataset(Dataset):
    """Combining EVERY piece from this notebook: tokenization + vocabulary
    (Chapter 47), producing token ID sequences ready for the Transformer."""

    def __init__(self, texts: list, labels: list, vocab: dict, max_len=30):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.texts[idx].lower().split()[:self.max_len]
        ids = [self.vocab.get(t, self.vocab["<UNK>"]) for t in tokens]
        ids += [self.vocab["<PAD>"]] * (self.max_len - len(ids))   # pad to a FIXED length here,
                                                                       # simpler than Chapter 26's per-batch
                                                                       # padding, at the cost of some
                                                                       # wasted computation on short sequences
        return torch.tensor(ids), self.labels[idx]


# Illustrative usage
example_vocab = {"<PAD>": 0, "<UNK>": 1, "great": 2, "terrible": 3, "movie": 4, "was": 5}
dataset = SentimentDataset(["great movie", "terrible movie was bad"], [1, 0], example_vocab, max_len=10)
loader = DataLoader(dataset, batch_size=2)
sample_ids, sample_labels = next(iter(loader))
print("Sample batch shape:", sample_ids.shape)
print("Sample IDs:\n", sample_ids)
print("Labels:", sample_labels)

# Standard training setup, directly reusing Chapter 23's professional training loop functions
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)   # Transformers typically use SMALLER LRs than CNNs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print("\nTraining setup ready -- train_one_epoch()/evaluate() from Chapter 23 apply UNCHANGED,")
print("since TransformerClassifier follows the exact same nn.Module + logits-out contract")
print("as every other model built throughout this course.")


Sample batch shape: torch.Size([2, 10])
Sample IDs:
 tensor([[2, 4, 0, 0, 0, 0, 0, 0, 0, 0],
        [3, 4, 5, 1, 0, 0, 0, 0, 0, 0]])
Labels: tensor([1, 0])

Training setup ready -- train_one_epoch()/evaluate() from Chapter 23 apply UNCHANGED,
since TransformerClassifier follows the exact same nn.Module + logits-out contract
as every other model built throughout this course.


### Common Mistakes
- Using an inappropriately large learning rate for a Transformer (borrowed from CNN-training intuition) — Transformers are often more sensitive to learning rate and typically trained with smaller LRs and warmup (Chapter 21) than CNNs commonly use.
- Padding every sequence to a large fixed `max_len` regardless of the actual batch's content — wastes computation on padding tokens; Chapter 26's per-batch dynamic padding is more efficient for datasets with highly variable sequence lengths, at some added implementation complexity.
- Using mean pooling over padded positions **without masking them out first** — the `<PAD>` tokens' (meaningless) embeddings would incorrectly dilute the mean; a correct implementation should mask padding positions before pooling (setting them to a very small value or explicitly excluding them from the average) — an easy, genuinely common bug to introduce.

### Interview Insight
**Q: You're building a Transformer-based text classifier and need to convert the encoder's per-token output sequence into a single vector for classification. What are your options, and what's the tradeoff?**
A: Common options: (1) mean pooling across the sequence dimension — simple, no extra parameters, but treats every token equally regardless of actual relevance to the classification task, and can be diluted by padding tokens if not masked correctly; (2) using only the final token's representation — relies on the architecture ensuring that position has aggregated whole-sequence context, which isn't automatically guaranteed without careful design; (3) a dedicated, learnable `[CLS]` token prepended to every sequence (BERT's approach, Chapter 53) — the model learns, via attention, what information to aggregate into that specific position, generally considered the most principled and highest-performing option, at the cost of a small amount of extra architectural complexity (adding and handling the special token). In practice, the `[CLS]`-token approach is standard for models built on top of pretrained Transformers; simple mean pooling remains a reasonable, easy baseline for smaller custom models trained from scratch.

### Exercise
Build the complete `TransformerClassifier` + `SentimentDataset` pipeline yourself, and correctly implement a masked mean-pooling variant that excludes `<PAD>` token positions from the average.

### Quiz (Chapter 52)
1. What does `encoded.mean(dim=1)` compute, and why `dim=1` specifically?
2. What's the standard, more principled alternative to mean pooling that modern pretrained Transformers use?
3. Why can naive mean pooling over padded sequences produce incorrect results, and how would you fix it?
4. Why might a Transformer need a smaller learning rate than a comparable CNN?

### Chapter 52 Summary
- A complete Transformer-based classifier combines: tokenization/vocabulary (Ch 47) → embeddings + positional encoding (Ch 48, 50) → stacked Transformer blocks (Ch 49-51) → pooling → classification head — every piece from this notebook assembled into one genuine, trainable system.
- Pooling strategy (mean, final-token, or `[CLS]`-token) is a real design choice with genuine tradeoffs, directly motivating BERT's specific architectural choice (Chapter 53).

---

## END OF NOTEBOOK 9 — What You Should Now Be Able To Do
- Choose between RNN, LSTM, and GRU deliberately based on task requirements, computational constraints, and data availability.
- Build a complete text preprocessing pipeline: tokenization, vocabulary construction (train-only, with `<PAD>`/`<UNK>`), encoding, padding, and attention masks.
- Explain and use `nn.Embedding`, and articulate why semantic similarity emerges purely from training, without being explicitly programmed.
- Implement scaled dot-product attention and multi-head attention from first principles, explaining every step's purpose (including the `sqrt(d_k)` scaling).
- Explain why Transformers need positional encoding, build a complete Transformer block with residual connections and LayerNorm, and correctly use `nn.ModuleList` for a stack of submodules.
- Assemble a complete, working Transformer-based text classifier end-to-end, including a deliberate pooling strategy choice.

## Before Moving to Notebook 10 (Chapters 53–58: Hugging Face + PyTorch, Large Language Models, Build a Mini Language Model, Text Generation, LLM Fine-Tuning, LoRA and PEFT)
Say **"Continue to the next notebook"** to proceed — this is where the course reaches large language models directly.
